# TrustFed-IoT — Benchmark Experiments (5 Laptops / 5 Emails)

**Before running:** Runtime → Change runtime type → **T4 GPU**

| Tab | Email | Shard | Time |
|-----|-------|-------|------|
| Tab 1 | email1@gmail.com | `--shard-index 0` | ~1.5 hrs |
| Tab 2 | email2@gmail.com | `--shard-index 1` | ~1.5 hrs |
| Tab 3 | email3@gmail.com | `--shard-index 2` | ~1.5 hrs |
| Tab 4 | email4@gmail.com | `--shard-index 3` | ~1.5 hrs |
| Tab 5 | email5@gmail.com | `--shard-index 4` | ~1.5 hrs |

**Total time: ~1.5 hours** (all running in parallel)

In [ ]:
#@title Step 1: Install dependencies
!pip install torch torchvision optuna requests matplotlib numpy tqdm

In [ ]:
#@title Step 2: Clone project from GitHub
#@markdown Replace YOUR_USERNAME with your GitHub username
import os

REPO_URL = "https://github.com/YOUR_USERNAME/trustfed-iot.git"  #@param {type:"string"}

!git clone $REPO_URL
PROJECT_DIR = REPO_URL.split("/")[-1].replace(".git", "")
os.chdir(PROJECT_DIR)
print(f"Switched to: {os.getcwd()}")

In [ ]:
#@title Step 3: Verify GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU! Go to Runtime -> Change runtime type -> GPU")

In [ ]:
#@title Step 4: Rebuild partition cache
!python -m data.build_partition_cache

In [ ]:
#@title Step 5: Run benchmark — choose your shard number!
#@markdown Set SHARD_NUMBER to 0, 1, 2, 3, or 4:
SHARD_NUMBER = 0  #@param [0, 1, 2, 3, 4]

!python -m experiments.run_all_experiments \
  --rounds 100 \
  --seeds 1 2 3 4 5 \
  --attacks clean gaussian sign_flip scaling label_flip \
  --methods proposed fedavg multikrum \
  --num-shards 5 --shard-index $SHARD_NUMBER \
  --export-zip

In [ ]:
#@title Step 6: Download YOUR shard result
!zip -r benchmark_shard_${SHARD_NUMBER}_of_5.zip \
  results/exports/benchmark_shard_${SHARD_NUMBER}_of_5.zip

from google.colab.files import download
download(f'benchmark_shard_{SHARD_NUMBER}_of_5.zip')

---
## After ALL 5 laptops/emails finish (~1.5 hours):

1. Download all 5 ZIP files
2. Copy them to ONE laptop
3. Run the merge cell below

In [ ]:
#@title Step 7: Merge all shards and generate plots (run on ONE laptop only)
#@markdown Upload all 5 ZIP files to results/exports/ first, then run this
!python -m experiments.plot_experiments \
  --root results/benchmark/ \
  --import-zip results/exports/benchmark_shard_0_of_5.zip \
  --import-zip results/exports/benchmark_shard_1_of_5.zip \
  --import-zip results/exports/benchmark_shard_2_of_5.zip \
  --import-zip results/exports/benchmark_shard_3_of_5.zip \
  --import-zip results/exports/benchmark_shard_4_of_5.zip

In [ ]:
#@title Step 8: Download final plots and summary
!zip -r final_results.zip \
  results/benchmark/figures/ \
  results/benchmark/summary/

from google.colab.files import download
download('final_results.zip')